# Unit 16 / Chapter 16: Fault-Tolerant Quantum AI

> **Main Learning Objective:** Understand what fault tolerance means for a quantum computer, how logical qubits are built from many noisy physical qubits, how big the resource overhead is, and what "quantum AI in the fault-tolerant era" will actually look like.

| Section | Topic |
|---|---|
| 16.1 | From NISQ to fault-tolerant: what changes |
| 16.2 | Logical qubits and the surface code threshold |
| 16.3 | Magic state distillation and the FT overhead |
| 16.4 | What quantum AI looks like on fault-tolerant hardware |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 16**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 16
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 16.1: From NISQ to Fault-Tolerant: What Changes

The devices we have today are **NISQ**: Noisy, Intermediate-Scale, Quantum. A few hundred qubits, error rates around 0.1 percent per gate, coherence times measured in tens or hundreds of microseconds. That is enough to demonstrate ideas but not enough to run a long quantum algorithm like Shor's factoring of RSA-2048 or a deep quantum neural network with millions of gates.

**Fault-tolerant quantum computing (FTQC)** changes this. The trick: encode a single "logical" qubit into a redundant block of many physical qubits, so that errors on physical qubits can be detected and corrected before they corrupt the logical information.

The catch: fault tolerance is not free. You pay in:

* **Extra qubits**: today's estimates put a useful logical qubit at 1000 to 10000 physical qubits.
* **Slower operations**: logical gates take many rounds of physical operations.
* **Distilled magic states**: certain gates (T gates, non-Clifford operations) require an expensive resource called a magic state, which must be prepared and purified.

In short: a fault-tolerant quantum computer is possible but massive. Today's estimates say Shor's algorithm on RSA-2048 requires around 20 million physical qubits. This is the mountain quantum AI must climb to run deep, error-free algorithms.

---
# Section 16.2: Logical Qubits and the Surface Code Threshold

The dominant error-correction code today is the **surface code**. Physical qubits sit on a 2D grid. Half the qubits store the quantum data, the other half constantly measure parity checks that reveal errors.

The magic property: if the physical error rate p per gate is **below a threshold** (around 1 percent for the surface code), then increasing the code size drives the logical error rate **exponentially** toward zero.

If p is above the threshold, adding more qubits actually makes things worse: you introduce errors faster than you can correct them. So the whole enterprise depends on staying below threshold.

Below we simulate the surface code's threshold behavior in a tiny toy model: logical error rate as a function of physical error rate for different code distances d.

In [ ]:
# Toy model of surface code threshold behavior:
# logical error ~ A * (p / p_th) ** ((d+1)/2), where p_th ~ 0.01
p_th = 0.01
p_range = np.logspace(-4, -1.3, 30)
distances = [3, 5, 7, 9]

fig, ax = plt.subplots(figsize=(6, 4))
for d in distances:
    logical = 0.1 * (p_range / p_th) ** ((d+1)/2)
    ax.loglog(p_range, logical, label=f"d = {d}")
ax.axvline(p_th, color='k', linestyle='--', alpha=0.5)
ax.set_xlabel("physical error rate p")
ax.set_ylabel("logical error rate")
ax.set_title("Surface code threshold behavior (toy model)")
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

print("At p = 0.001 (below threshold):")
for d in distances:
    print(f"  d = {d}: logical error ~ {0.1 * (0.001 / p_th) ** ((d+1)/2):.2e}")
print("\nAt p = 0.02 (above threshold):")
for d in distances:
    print(f"  d = {d}: logical error ~ {0.1 * (0.02 / p_th) ** ((d+1)/2):.2e}")

Below the threshold, doubling the code distance shrinks the logical error by orders of magnitude. Above the threshold, larger codes fail faster than small ones. This is why hitting 1 percent physical error rates is such a widely-quoted milestone: it is the price of entry for FTQC.

### Activity 16.1

For a target logical error rate of 1e-9 (roughly what Shor's algorithm needs), estimate the minimum code distance d you would need when the physical error rate is p = 0.001. Use the toy formula.

In [ ]:
# TODO: solve 0.1 * (p / p_th) ** ((d+1)/2) = 1e-9 for d, when p = 0.001.
p = 0.001
target = 1e-9
p_th = 0.01

# Try d values and pick the smallest that meets the target
required_d = None
for d in range(3, 40, 2):
    logical = 0.1 * (p / p_th) ** ((d+1)/2)
    if logical <= target:
        required_d = d
        break

print(f"Minimum required distance: d = {required_d}")
# Each logical qubit uses about 2*d*d physical qubits in the surface code
print(f"Physical qubits per logical qubit at this distance: ~{2*required_d*required_d}")

<details><summary>Solution</summary>

Solving `(0.001 / 0.01)^((d+1)/2) = 1e-8`, i.e. `10^(-(d+1)/2) = 10^(-8)`, gives `(d+1)/2 = 8`, so d = 15. That corresponds to about 450 physical qubits per logical qubit. This is why estimates for a full FTQC RSA-2048 factorization run into the 20 million qubit range.
</details>

---
# Section 16.3: Magic State Distillation and the FT Overhead

The surface code only protects a limited set of gates for free: the so-called Clifford gates (H, S, CNOT). But Clifford-only circuits are classically simulatable, so they cannot achieve any speedup on their own.

You need at least one non-Clifford gate. The standard choice is the **T gate**. But T gates cannot be done directly on encoded surface-code qubits, and instead require a pre-prepared **magic state** consumed via gate teleportation.

Magic states are prepared noisily and then **distilled**: many noisy magic states are combined by a distillation circuit that outputs fewer, cleaner ones. A famous distillation protocol takes 15 noisy magic states and produces 1 magic state whose error is cubed. Repeat, and errors drop fast.

The cost of magic state distillation dominates the resource estimates for real algorithms. A single call to HHL or Shor at production scale needs billions of distilled T states.

In [ ]:
# Toy model: recursive magic state distillation.
# One 15-to-1 round takes noisy states of error p to output error 35 p^3 (approximate).
def distill_once(p_in):
    return 35 * p_in**3

p_noisy = 1e-3
history = [p_noisy]
for r in range(1, 6):
    p_noisy = distill_once(p_noisy)
    history.append(p_noisy)
    print(f"after round {r}: error = {p_noisy:.3e}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(history, marker='o')
ax.set_xlabel("distillation round"); ax.set_ylabel("residual error")
ax.set_title("Magic state distillation, 15-to-1 protocol")
ax.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()

### Activity 16.2

How many rounds of distillation are needed to bring an initial error of 1e-2 below 1e-15?

In [ ]:
# TODO: iterate distill_once until error is below 1e-15, count the rounds.
p = 1e-2
rounds = None

r = 0
while p > 1e-15:
    p = distill_once(p)
    r += 1
    if r > 20: break
rounds = r
print(f"rounds needed: {rounds}, final error {p:.2e}")

<details><summary>Solution</summary>

Each round takes error e to about 35 * e^3. Starting from 1e-2: round 1 gives ~3.5e-5, round 2 gives ~1.5e-12, round 3 gives ~1.2e-34. So 3 rounds are enough for the 1e-15 target. Each round costs 15x more magic states than the previous, so overheads pile up fast.
</details>

---
# Section 16.4: What Quantum AI Looks Like on Fault-Tolerant Hardware

Today's quantum ML is stuck at shallow, noisy circuits with heuristic ansatze. Fault-tolerant quantum AI will look very different:

* **Deep, structured algorithms** with provable speedups replace variational heuristics. Think HHL for large linear systems, Grover-boosted search over huge model spaces, quantum PCA on exponentially large feature spaces.
* **Precise chemistry and materials simulation** turns quantum AI into a design tool: predict a molecule's properties without measuring anything in a lab.
* **Cryptography-safe machine learning** where privacy is enforced by physics, not encryption.
* **Long quantum reservoirs** and **quantum-enhanced sampling** for generative models with genuinely intractable-classically distributions.

The scale problem is the ticket: everything in this list is impossible today, and will be run-of-the-mill on a mature fault-tolerant machine.

Below we compute a rough "resource estimate" for a hypothetical FT quantum kernel over 1000 features.

In [ ]:
# Rough resource estimate for a FT quantum kernel evaluation
n_features = 1000
qubits_needed = int(np.ceil(np.log2(n_features)))
t_gates_per_kernel = 10_000_000        # rough
distilled_states = t_gates_per_kernel  # 1 magic state per T gate
distillation_ratio = 15**3             # 3 rounds
raw_magic_states = distilled_states * distillation_ratio
logical_qubits = qubits_needed + 50    # ancillas
physical_per_logical = 2 * 15**2       # from earlier

print(f"qubits (address only):        {qubits_needed}")
print(f"logical qubits total:         {logical_qubits}")
print(f"physical qubits total:        {logical_qubits * physical_per_logical:,}")
print(f"raw magic states needed:      {raw_magic_states:,}")
print("Compare to today's ~1000 physical qubit systems: FT quantum AI is a decade or more away.")

### Activity 16.3

In one sentence, explain why fault-tolerant quantum AI is likely to arrive *later* than fault-tolerant Shor factoring, despite QML being seen as a shorter-term goal.

In [ ]:
answer_16_3 = """YOUR ANSWER HERE, one or two sentences."""
print(answer_16_3)

<details><summary>Sample answer</summary>

Shor's algorithm has a clear, well-studied circuit whose FT cost is known. QML algorithms with provable exponential speedups are much less understood: many claimed speedups have been dequantized, so we do not yet know which QML tasks are worth spending FT resources on. That makes cryptographic threats a nearer target than QML deployment.
</details>

---
## Section summary

* NISQ hardware is noisy and shallow; FTQC uses error correction to run deep, exact quantum algorithms.
* The surface code exponentially suppresses logical errors below its threshold, but costs many physical qubits per logical qubit.
* T gates require distilled magic states, and distillation costs dominate the resource budget.
* FT quantum AI unlocks deep, provably-fast algorithms, but is likely a decade or more away.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** NISQ stands for:

A. Native In-Silicon Quantum
B. Noisy, Intermediate-Scale, Quantum
C. Non-Interacting Superconducting Qubits
D. Near-Ideal Superposition Qubits

**Q2.** The surface code protects information by:

A. Encoding data in a 2D lattice with constant parity checks
B. Cooling qubits below 1 mK
C. Using a giant amplifier
D. Deleting bad qubits from the register

**Q3.** Below the surface-code threshold, doubling the code distance:

A. Doubles the logical error rate
B. Exponentially reduces the logical error rate
C. Has no effect
D. Removes the need for measurement

**Q4.** Above the surface-code threshold, adding more qubits:

A. Reduces errors even faster
B. Makes things worse; errors accumulate faster than they can be corrected
C. Has no effect
D. Turns the system classical

**Q5.** The surface code protects which gate set for free?

A. Only single-qubit gates
B. Clifford gates (H, S, CNOT)
C. Only T gates
D. All gates

**Q6.** Why do you need magic state distillation?

A. To cool the qubits
B. To perform non-Clifford gates like T, which are needed for any true speedup
C. To read out the qubits
D. To connect qubits by fiber

**Q7.** The 15-to-1 magic state distillation protocol takes error p to approximately:

A. 15p
B. p/15
C. 35 p^3
D. 2p

**Q8.** For a target logical error of 1e-9 with physical error 1e-3, roughly what code distance did we compute?

A. d = 3
B. d = 7
C. d = 15
D. d = 100

**Q9.** Which of the following is NOT a natural fit for FT quantum AI?

A. HHL for large linear systems
B. Quantum-enhanced sampling for generative models
C. Deep quantum kernel over 10000 features
D. Running a 4-qubit variational ansatz

**Q10.** Roughly when do experts commonly say practical FT quantum computers might arrive?

A. 2026
B. 2028
C. Around 2035 or later, though estimates vary widely
D. 2050 minimum

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 16!")